# 04 — Validation: July 2020 net surface shortwave vs. CERES EBAF

Compare the model's monthly-mean net downward surface shortwave radiation
($Q_{sw}$, `oceQsw`) for July 2020 against the CERES EBAF Ed4.2 surface product
(all-sky surface net shortwave) on a common 1° grid.

**Design choices.** This is a free-running coupled nature run, so the comparison is
statistical (monthly mean, zonal means, area-weighted bias/RMSE) — never instantaneous
fields, whose cloud positions cannot match observations. The model mean is built from
3-hourly instantaneous output (8 fixed local solar times per longitude — adequate diurnal
sampling; set `SUBSAMPLE = 1` for the full hourly record at ~3× the I/O). Statistics are
restricted to the open ocean between 60°S–60°N: poleward of that, sea-ice albedo makes
the surface net SW definitions diverge between model coupler and satellite retrieval.

**Obtaining the CERES file** (one-time, ~10 MB):
1. Go to https://ceres.larc.nasa.gov/data/ → *EBAF* → "Order Data".
2. Select **EBAF Ed4.2**, *Surface Fluxes* → **Net Shortwave Flux, All-Sky**
   (variable `sfc_net_sw_all_mon`), temporal range covering **July 2020**, global region.
3. Download the NetCDF subset and upload it next to this notebook (JupyterLab: drag &
   drop), then point `CERES_FILE` below at it.

Reference values for orientation: global-mean all-sky surface net SW is ≈ 160–165 W m⁻²;
regional monthly biases within ±10–20 W m⁻² are typical of well-performing models, with
classic error patterns under the subtropical stratocumulus decks and over the Southern
Ocean. Cite CERES EBAF (Loeb et al.; Kato et al.) when publishing — [citation to be
inserted].

In [ ]:
# Environment check: run on SciServer (Kraken domain, with the Poseidon DYAMOND
# ceph volume attached), or set DYAMOND_ROOT to a local subset.
from dyamond_fluxes import dyamond_root

root = dyamond_root()
print(f"DYAMOND root: {root}")

In [ ]:
from pathlib import Path

# --- configuration -------------------------------------------------------
MONTH_START, MONTH_END = "2020-07-01", "2020-08-01"  # [start, end)
SUBSAMPLE = 3            # take every 3rd hourly dump; 1 = all hourly output
DLON = DLAT = 1.0        # comparison grid (matches CERES 1 deg)
CERES_FILE = Path("CERES_EBAF_Ed4.2_Subset_202007-202007.nc")  # adjust to your download
CERES_VAR_CANDIDATES = ["sfc_net_sw_all_mon", "sfc_net_sw_all", "sfc_net_sw"]

FIGDIR = Path("../figures")
FIGDIR.mkdir(exist_ok=True)
CACHE = Path(f"qsw_model_{MONTH_START[:7]}_1deg.nc")  # gitignored (*.nc)

In [ ]:
# Dask cluster for the ~60 GB (SUBSAMPLE=3) of file reads behind the time mean.
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=4, threads_per_worker=2, memory_limit="8GB")
client = Client(cluster)
client

## Model: July-2020 mean $Q_{sw}$ on the 1° grid

Time-mean first on the native grid (a dask reduction over one ~243 MB file per kept
time step), then area-weighted binning to 1°. The result is cached to NetCDF so reruns
and later notebooks skip the expensive pass.

In [ ]:
import xarray as xr

from dyamond_fluxes import bin_to_latlon, open_ocean_dataset, to_positive_down

if CACHE.exists():
    qsw_model = xr.open_dataarray(CACHE)
    print(f"loaded cached model mean from {CACHE}")
else:
    ds = open_ocean_dataset(["oceQsw"])
    qsw = ds["oceQsw"].sel(time=slice(MONTH_START, MONTH_END))
    qsw = qsw.isel(time=slice(None, None, SUBSAMPLE))
    print(f"averaging {qsw.sizes['time']} time steps:",
          qsw.time.values[0], "to", qsw.time.values[-1])

    qsw_mean = to_positive_down(qsw).mean("time")          # lazy reduction
    qsw_mean = qsw_mean.where(ds["Depth"] > 0).load()      # ocean only; compute here

    qsw_model = bin_to_latlon(
        qsw_mean, ds["XC"], ds["YC"], area=ds["rA"], dlon=DLON, dlat=DLAT
    )
    qsw_model.name = "qsw_model"
    qsw_model.attrs.update(
        long_name=f"model net downward surface shortwave, mean {MONTH_START[:7]}",
        units="W m-2",
        subsample=f"every {SUBSAMPLE} hourly dump(s)",
    )
    qsw_model.to_netcdf(CACHE)
    print(f"cached to {CACHE}")
qsw_model

## CERES EBAF surface net shortwave, July 2020

CERES uses 0–360° longitudes and its own variable naming; wrap to −180–180° and align
with the model's bin centers (both grids are 1° with centers at ±0.5°).

In [ ]:
import numpy as np

if not CERES_FILE.exists():
    raise FileNotFoundError(
        f"{CERES_FILE} not found. Order the EBAF Ed4.2 surface subset from "
        "https://ceres.larc.nasa.gov/data/ (see the notebook header) and upload it here."
    )

ceres_ds = xr.open_dataset(CERES_FILE)
var = next((v for v in CERES_VAR_CANDIDATES if v in ceres_ds), None)
if var is None:
    raise KeyError(
        f"None of {CERES_VAR_CANDIDATES} found; available: {list(ceres_ds.data_vars)}"
    )

qsw_ceres = ceres_ds[var].sel(time=MONTH_START[:7]).squeeze()
# wrap longitudes to [-180, 180) and sort to match the model grid
qsw_ceres = qsw_ceres.assign_coords(lon=(((qsw_ceres.lon + 180) % 360) - 180)).sortby("lon")
qsw_ceres = qsw_ceres.reindex_like(qsw_model, method="nearest", tolerance=0.51)
print(f"CERES variable: {var}; global mean {float(qsw_ceres.mean()):.1f} W m-2")

## Maps: model, CERES, and bias

In [ ]:
import cmocean
import matplotlib.pyplot as plt

ocean = qsw_model.notnull()  # model land/ice-masked bins define the comparison domain

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True, sharey=True)
for ax, (da, label) in zip(
    axes,
    [(qsw_model, "GEOS-MITgcm DYAMOND"), (qsw_ceres.where(ocean), "CERES EBAF Ed4.2")],
):
    pc = ax.pcolormesh(da.lon, da.lat, da, cmap=cmocean.cm.thermal, vmin=0, vmax=320)
    ax.set_title(f"{label} — net surface shortwave, July 2020")
fig.colorbar(pc, ax=axes, shrink=0.8, label="W m$^{-2}$")
fig.savefig(FIGDIR / "qsw_validation_maps.png", dpi=200, bbox_inches="tight")

In [ ]:
bias = (qsw_model - qsw_ceres).where(ocean)

fig, ax = plt.subplots(figsize=(11, 4.5))
pc = ax.pcolormesh(bias.lon, bias.lat, bias, cmap=cmocean.cm.balance, vmin=-60, vmax=60)
fig.colorbar(pc, ax=ax, label="W m$^{-2}$")
ax.set_title("Model minus CERES EBAF, net surface shortwave, July 2020")
fig.savefig(FIGDIR / "qsw_validation_bias.png", dpi=200, bbox_inches="tight")

## Zonal means and area-weighted statistics (ocean, 60°S–60°N)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
qsw_model.mean("lon").plot(y="lat", ax=ax, label="model")
qsw_ceres.where(ocean).mean("lon").plot(y="lat", ax=ax, label="CERES EBAF")
ax.set_xlabel("W m$^{-2}$")
ax.set_ylabel("latitude")
ax.set_title("Zonal-mean net surface SW, July 2020 (ocean)")
ax.legend()
ax.grid(alpha=0.3)
fig.savefig(FIGDIR / "qsw_validation_zonal.png", dpi=200, bbox_inches="tight")

In [ ]:
# Area weighting: bin areas on a regular lat-lon grid scale with cos(lat).
w = np.cos(np.deg2rad(qsw_model.lat)).broadcast_like(qsw_model)
inband = ocean & (abs(qsw_model.lat) < 60) & qsw_ceres.notnull()

def wstat(da):
    return float(da.where(inband).weighted(w.where(inband, 0.0)).mean())

bias_mean = wstat(qsw_model - qsw_ceres)
rmse = np.sqrt(wstat((qsw_model - qsw_ceres) ** 2))
print(f"model ocean mean (60S-60N):  {wstat(qsw_model):7.1f} W m-2")
print(f"CERES ocean mean (60S-60N):  {wstat(qsw_ceres):7.1f} W m-2")
print(f"mean bias (model - CERES):   {bias_mean:7.1f} W m-2")
print(f"RMSE of 1-deg monthly bins:  {rmse:7.1f} W m-2")

## Interpretation guide

- **Mean bias within ±10 W m⁻²** and RMSE of order 15–25 W m⁻² for 1° monthly bins would
  place the simulation among well-performing coupled models for surface SW.
- **Positive bias under the stratocumulus decks** (SE Pacific, SE Atlantic, off
  California) is the classic too-few/too-thin low clouds signature; check whether the
  bias map is dominated by these regions.
- **A uniform offset** (same sign everywhere, including clear-sky regions) points to a
  systematic difference — aerosol/clear-sky transmission, ocean albedo treatment, or a
  residual sampling artifact — rather than cloud errors; reduce `SUBSAMPLE` to 1 to rule
  out diurnal sampling.
- Remember `oceQsw` is what the coupler delivers to the ocean; CERES estimates the flux
  at the air–sea interface. Penetrating-SW handling and ice-edge cells account for
  edge-of-domain differences, which is why statistics stop at 60°.

Next validation steps: repeat for a boreal-winter month (January 2021); compare the
diurnal-cycle composite against CERES SYN1deg-1H; point comparisons against the
TAO/PIRATA/RAMA buoy downwelling SW (with an albedo factor ≈ 0.94) or the OceanSITES
flux reference stations.